In [1]:
import pandas as pd
import numpy as np


In [2]:
FEATURE_PATH = "../Data/Feature_Data"

enrol_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_enrolment.csv",
    parse_dates=["date"]
)

demo_bio_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_demo_bio_combined.csv",
    parse_dates=["date"]
)


In [3]:
sort_keys = ["state_clean", "district_clean", "pincode", "date"]

enrol_feat = enrol_feat.sort_values(sort_keys)
demo_bio_feat = demo_bio_feat.sort_values(sort_keys)


In [4]:
enrol_feat["enrolment_dod_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .diff()
)

enrol_feat["enrolment_dod_pct_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .pct_change()
)


In [5]:
enrol_feat["enrolment_7day_mean"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

enrol_feat["enrolment_7day_std"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).std())
)


In [6]:
enrol_feat["enrolment_volatility"] = np.where(
    enrol_feat["enrolment_7day_mean"] > 0,
    enrol_feat["enrolment_7day_std"] / enrol_feat["enrolment_7day_mean"],
    0
)


In [7]:
enrol_feat["sudden_surge_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] > 0.5
).astype(int)

enrol_feat["sudden_drop_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] < -0.5
).astype(int)


In [8]:
demo_bio_feat["bio_demo_stress_ratio"] = np.where(
    demo_bio_feat["total_demographic_updates"] > 0,
    demo_bio_feat["total_biometric_updates"] /
    demo_bio_feat["total_demographic_updates"],
    0
)


In [9]:
demo_bio_feat["high_biometric_stress_flag"] = (
    demo_bio_feat["bio_demo_stress_ratio"] > 1.5
).astype(int)


In [10]:
import os

ADV_FEATURE_PATH = "../Data/Advanced_Feature_Data"
os.makedirs(ADV_FEATURE_PATH, exist_ok=True)


In [11]:
enrol_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_enrolment.csv",
    index=False
)

demo_bio_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_demo_bio.csv",
    index=False
)


In [12]:
import os
os.listdir("../Data/Advanced_Feature_Data")


['advanced_feature_demo_bio.csv', 'advanced_feature_enrolment.csv']

In [13]:
import pandas as pd


In [14]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [15]:
ADV_FEATURE_PATH = "../Data/Advanced_Feature_Data"


In [16]:
adv_enrol_df = pd.read_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_enrolment.csv",
    parse_dates=["date"]
)

adv_demo_bio_df = pd.read_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_demo_bio.csv",
    parse_dates=["date"]
)


In [17]:
print("Advanced Enrolment Features — HEAD")
print(adv_enrol_df.head())


Advanced Enrolment Features — HEAD
        date                  state_clean district_clean  pincode  age_0_5  age_5_17  age_18_greater  year  month  \
0 2025-09-01  Andaman and Nicobar Islands       andamans   744101        0         1               0  2025      9   
1 2025-09-04  Andaman and Nicobar Islands       andamans   744101        1         0               0  2025      9   
2 2025-09-17  Andaman and Nicobar Islands       andamans   744101        1         0               0  2025      9   
3 2025-09-18  Andaman and Nicobar Islands       andamans   744101        1         0               0  2025      9   
4 2025-09-26  Andaman and Nicobar Islands       andamans   744101        1         0               0  2025      9   

   quarter  week_of_year  day_of_week  is_weekend  total_enrolments  child_enrolments  adult_ratio  child_share  \
0        3            36            0           0                 1                 1          0.0          1.0   
1        3            36        

In [18]:
print("\nAdvanced Enrolment Features — DESCRIBE")
print(adv_enrol_df.describe())



Advanced Enrolment Features — DESCRIBE
                                date        pincode        age_0_5       age_5_17  age_18_greater      year  \
count                         994925  994925.000000  994925.000000  994925.000000   994925.000000  994925.0   
mean   2025-10-23 18:43:59.111089408  516327.986734       3.514702       1.712573        0.166796    2025.0   
min              2025-03-02 00:00:00  110001.000000       0.000000       0.000000        0.000000    2025.0   
25%              2025-09-19 00:00:00  362275.000000       1.000000       0.000000        0.000000    2025.0   
50%              2025-10-27 00:00:00  516107.000000       2.000000       0.000000        0.000000    2025.0   
75%              2025-11-15 00:00:00  695009.000000       3.000000       1.000000        0.000000    2025.0   
max              2025-12-31 00:00:00  855456.000000    2688.000000    1812.000000      855.000000    2025.0   
std                              NaN  205571.461874      17.592445      

In [19]:
print("\nAdvanced Demographic + Biometric Features — DESCRIBE")
print(adv_demo_bio_df.describe())



Advanced Demographic + Biometric Features — DESCRIBE
                                date       pincode  demo_age_5_17  demo_age_17_  year_demo    month_demo  \
count                        1631503  1.631503e+06   1.631503e+06  1.631503e+06  1631503.0  1.631503e+06   
mean   2025-10-30 08:04:46.428404736  5.237261e+05   2.771920e+00  2.493287e+01     2025.0  1.054022e+01   
min              2025-03-01 00:00:00  1.100010e+05   0.000000e+00  0.000000e+00     2025.0  3.000000e+00   
25%              2025-09-20 00:00:00  3.911700e+05   0.000000e+00  3.000000e+00     2025.0  9.000000e+00   
50%              2025-11-05 00:00:00  5.226160e+05   1.000000e+00  7.000000e+00     2025.0  1.100000e+01   
75%              2025-12-04 00:00:00  6.905260e+05   2.000000e+00  1.700000e+01     2025.0  1.200000e+01   
max              2025-12-29 00:00:00  8.554560e+05   2.690000e+03  1.616600e+04     2025.0  1.200000e+01   
std                              NaN  1.995752e+05   1.671248e+01  1.399858e+02   